# Weather Forecasting TinyLlama LoRA on Colab

This notebook is a Colab-oriented runbook for the project training path. A T4 GPU is sufficient for TinyLlama LoRA when batch size, gradient accumulation, and sample count are conservative. It is not optimal for long full-dataset sweeps; A10/L4/A100-class GPUs will finish faster and tolerate larger batches.

In [ ]:
!nvidia-smi
!git clone https://github.com/ashioyajotham/weather_forecasting_lora.git
%cd weather_forecasting_lora

In [ ]:
!python -m pip install -U pip
!python -m pip install -r requirements.txt
!python -m pip install bitsandbytes

In [ ]:
import os
os.environ['USE_TF'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WEATHER_LORA_REPORT_TO'] = 'none'

# T4-friendly first pass. Increase after a successful smoke run.
os.environ['WEATHER_LORA_MAX_SAMPLES'] = '1000'

In [ ]:
!python scripts/smoke_check.py
!python scripts/eval_smoke.py
!python scripts/ppo_smoke.py
!python scripts/generation_quality_eval.py

In [ ]:
!python train_lora_peft.py

In [ ]:
!python merge_lora.py

## PPO

Run the dry run first. Start PPO only on a CUDA runtime with enough free memory. The local Windows CPU path is expected to reject PPO training because TRL value-head models do not support CPU/disk offload.

In [ ]:
!python train_ppo.py
# After confirming GPU headroom:
# !WEATHER_PPO_MAX_SAMPLES=64 python train_ppo.py --run-training --limit 64

## Artifacts

Download or mount Drive for `models/weather-lora-peft/lora_adapter` and `models/weather-merged`. GGUF conversion can be run locally if llama.cpp is available, or in Colab after installing llama.cpp conversion dependencies.